In [19]:
# 单元 1：基础导入与配置
import os
import sqlite3
import pandas as pd
import numpy as np

def calculate_tet(filepath, time_step=0.1):  
    try:  
        conn = sqlite3.connect(filepath)  
        
        df = pd.read_sql_query("SELECT * FROM trajectory_data", conn)  
        conn.close()  
        
        if 'frame' not in df.columns:  
            print(f"警告：{filepath} 缺少 'frame' 字段")  
            return np.nan  
        
        tet = df['frame'].max() * time_step
        
        return tet  
    
    except Exception as e:  
        print(f"处理 {filepath} 时发生错误: {e}")  
        return np.nan  

def process_small_tet():  
    base_path = "Small_Average"  
    layouts = ["A", "B", "C", "A2", "B2", "C2"]  
    widths = [round(w * 0.2 + 1.0, 1) for w in range(0, 8)]  # 1.0 - 2.4 m  
    runs = list(range(5))  
    
    results = {layout: {} for layout in layouts}  
    
    for layout in layouts:  
        layout_path = os.path.join(base_path, layout)  
        for width in widths:  
            tet_runs = []  
            for run in runs:
                filename = f"Width_{width:.1f}_{run}.sqlite"  
                filepath = os.path.join(layout_path, filename)  
                
                if os.path.exists(filepath):  
                    tet = calculate_tet(filepath)  
                    tet_runs.append(tet)  
                else:  
                    print(f"文件不存在: {filepath}")  
            
            # 计算该 width 下的 TET 均值  
            results[layout][width] = np.mean(tet_runs) if tet_runs else np.nan  
    
    # 转换为 DataFrame  
    df_tet = pd.DataFrame(results).round(2)  
    
    # 重新设置索引为宽度值  
    df_tet.index = widths  
    
    # 保存 CSV  
    os.makedirs("data/processed", exist_ok=True)  
    df_tet.to_csv("data/processed/small_tet_mean.csv")  
    
    return df_tet  

In [20]:

# 单元 4：执行并展示结果  
df_result = process_small_tet()  
print(df_result)  

         A      B      C     A2     B2     C2
1.0  30.72  42.30  33.70  19.72  19.68  16.58
1.2  30.42  37.00  30.74  17.04  19.94  15.22
1.4  27.80  29.96  27.22  16.52  18.70  14.46
1.6  26.42  26.66  28.04  16.02  19.00  14.34
1.8  25.28  23.46  25.66  15.80  19.26  14.42
2.0  22.74  23.24  25.38  15.66  18.66  14.34
2.2  21.50  23.22  24.96  15.48  17.80  13.72
2.4  21.54  22.94  21.38  15.50  17.52  13.84


In [5]:
def diagnose_data_loading():
    base_path = "Small_Average"
    layouts = ["A", "B", "C", "A2", "B2", "C2"]
    widths = [round(w * 0.2 + 0.8, 1) for w in range(0, 9)]
    
    print("检查目录结构:")
    for layout in layouts:
        layout_path = os.path.join(base_path, layout)
        print(f"{layout} 目录存在: {os.path.exists(layout_path)}")
        if os.path.exists(layout_path):
            files = os.listdir(layout_path)
            print(f"{layout} 目录文件数: {len(files)}")
            print(f"文件示例: {files[:3] if files else '无文件'}")

    print("\n检查文件命名:")
    for layout in layouts:
        layout_path = os.path.join(base_path, layout)
        if os.path.exists(layout_path):
            for width in widths:
                filename = f"Width_{width:.1f}_0.sqlite"
                filepath = os.path.join(layout_path, filename)
                print(f"{filepath} 文件存在: {os.path.exists(filepath)}")

diagnose_data_loading()


检查目录结构:
A 目录存在: True
A 目录文件数: 42
文件示例: ['Width_0.8.sqlite', 'Width_0.8_0.sqlite', 'Width_1.0_0.sqlite']
B 目录存在: True
B 目录文件数: 40
文件示例: ['Width_1.0_0.sqlite', 'Width_1.0_1.sqlite', 'Width_1.0_2.sqlite']
C 目录存在: True
C 目录文件数: 40
文件示例: ['Width_1.0_0.sqlite', 'Width_1.0_1.sqlite', 'Width_1.0_2.sqlite']
A2 目录存在: True
A2 目录文件数: 40
文件示例: ['Width_1.0_0.sqlite', 'Width_1.0_1.sqlite', 'Width_1.0_2.sqlite']
B2 目录存在: True
B2 目录文件数: 40
文件示例: ['Width_1.0_0.sqlite', 'Width_1.0_1.sqlite', 'Width_1.0_2.sqlite']
C2 目录存在: True
C2 目录文件数: 40
文件示例: ['Width_1.0_0.sqlite', 'Width_1.0_1.sqlite', 'Width_1.0_2.sqlite']

检查文件命名:
Small_Average\A\Width_0.8_0.sqlite 文件存在: True
Small_Average\A\Width_1.0_0.sqlite 文件存在: True
Small_Average\A\Width_1.2_0.sqlite 文件存在: True
Small_Average\A\Width_1.4_0.sqlite 文件存在: True
Small_Average\A\Width_1.6_0.sqlite 文件存在: True
Small_Average\A\Width_1.8_0.sqlite 文件存在: True
Small_Average\A\Width_2.0_0.sqlite 文件存在: True
Small_Average\A\Width_2.2_0.sqlite 文件存在: True
Small_Average\A\Width_2